#web Scraping....  niketPractice + ExtraInformation 

In [ ]:
from bs4 import BeautifulSoup
import requests
url = 'http://www.prnewswire.com/news-releases/tata-consultancy-services-reports-broad-based-growth-across-markets-marks-steady-fy17-300440934.html'
r = requests.get(url)
html = r.text
#soup is like Entire Dom ...
soup = BeautifulSoup(html, 'html.parser')
links = soup.get_text()
print(links)


In [ ]:
from bs4 import BeautifulSoup
import requests
url = 'http://www.prnewswire.com/news-releases/tata-consultancy-services-reports-broad-based-growth-across-markets-marks-steady-fy17-300440934.html'
r = requests.get(url)
html = r.text 
#print(html)
soup = BeautifulSoup(html, 'html.parser')
# print(soup)
links = [ e.get_text() for e in soup.find_all('p')   ]
print(links)
#learning... 
article = '\n'.join(links)

'''
#selenium(browser automation tool)
Why we need it?

current tools:
requests → just downloads HTML 
BeautifulSoup → parses HTML 

Problem:
Some websites load content using JavaScript after page loads
So:
requests gets incomplete HTML
You miss real content

Selenium solves this:
opens real browser
waits for JS to load
then gives full content

there is one more thing #UI_PATH(will explore further)
'''

In [ ]:
urls = ['http://www.cs.bham.ac.uk/research/groupings/machine-learning/',
'http://www.cs.bham.ac.uk/research/groupings/robotics/',
'http://www.cs.bham.ac.uk/research/groupings/reasoning/']

from bs4 import BeautifulSoup
import requests
from pprint import pprint

for url in urls:
    soup = BeautifulSoup(requests.get(url).text, 'lxml')

    # parse title
    title = soup.select_one('h1.title').text

    # parse academic staff (if any):
    staff_list = []
    if soup.select('h2 ~ ul'):
        for li in soup.select('h2 ~ ul')[-1].find_all('li'):
            staff_list.append(li.text)
            li.clear()
        soup.select('h2')[-1].clear()

    # parse the text
    text = ''
    for t in soup.select('nav ~ *'):
        text += t.text.strip() + '\n'

    print(title)
    print(text)
    print('Staff list = ', staff_list)
    print('-' * 80)


'''
Short and clear:
BeautifulSoup (Parser)
if web is static and following a structure format
eg('http://www.prnewswire.com/news-releases/tata-consultancy-services-reports-broad-based-growth-across-markets-marks-steady-fy17-300440934.html')
👉 extracts data from HTML
👉 you control what to pick

Unstructured (Preprocessor)
eg('https://www.birmingham.ac.uk/about/college-of-engineering-and-physical-sciences/computer-science')
👉 cleans + formats data for LLM
👉 gives ready-to-use text

Selenium (Automation)
👉 opens browser, loads JS
👉 used when data not visible
UiPath (RPA)
👉 automates full workflows (click, type, etc.)
'''

'''
Scrapping using Unstructured
UnstructuredURLLoader
UnstructuredURLLoader of Langchain internally uses unstructured python library to load the content from url's

https://unstructured-io.github.io/unstructured/introduction.html

https://pypi.org/project/unstructured/#description

'''

In [9]:
#installing necessary libraries, libmagic is used for file type detection
!pip install langchain-community
!pip3 install unstructured libmagic python-magic python-magic-bin



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from langchain_community.document_loaders import UnstructuredURLLoader
loader = UnstructuredURLLoader(
    urls = ['http://www.cs.bham.ac.uk/research/groupings/robotics/']
)
docs = loader.load() 

print(docs)
# print(type(docs))
# print(len(docs))
# print(docs[0].page_content)

[Document(metadata={'source': 'http://www.cs.bham.ac.uk/research/groupings/robotics/'}, page_content='Skip to main content\n\nSelect campus:UKDubai\n\n\n\nUniversity of Birmingham\n\nMenu\n\nA graphical representation of a super computer\n\nSchool of Computer Science\n\nA graphical representation of a super computer\n\nBreaking new ground in the theory and practice of computational systems and their applications, the School of Computer Science is a progressive, inclusive department, providing specialist teaching and conducting world-leading research in fundamental and applied computer science.\n\nStudy with us\n\nAbout Computer Science\n\nLearn about our school\n\nPeople\n\nView our profiles\n\nUniversity of Birmingham\n\nEdgbaston\n\nBirmingham, B15 2TT\n\nUnited Kingdom\n\nTel:+44 (0)121 414 3344\n\nUniversity of Birmingham Instagram page\n\nUniversity of Birmingham LinkedIn page\n\nUniversity of Birmingham Twitter page\n\nUniversity of Birmingham Facebook page\n\nUniversity of Birmi

In [15]:
print(type(docs))

<class 'list'>


In [16]:
print(len(docs))

1


In [45]:
pageContent = docs[0].page_content
print(pageContent)

Skip to main content

Select campus:UKDubai



University of Birmingham

Menu

A graphical representation of a super computer

School of Computer Science

A graphical representation of a super computer

Breaking new ground in the theory and practice of computational systems and their applications, the School of Computer Science is a progressive, inclusive department, providing specialist teaching and conducting world-leading research in fundamental and applied computer science.

Study with us

About Computer Science

Learn about our school

People

View our profiles

University of Birmingham

Edgbaston

Birmingham, B15 2TT

United Kingdom

Tel:+44 (0)121 414 3344

University of Birmingham Instagram page

University of Birmingham LinkedIn page

University of Birmingham Twitter page

University of Birmingham Facebook page

University of Birmingham TikTok page

University of Birmingham Weibo page

University of Birmingham We Chat

University of Birmingham YouTube page

Culture and collect

In [20]:
docs[0].metadata

{'source': 'http://www.cs.bham.ac.uk/research/groupings/robotics/'}

textSplitters
Why do we need text splitters in first place?
LLM's have token limits. Hence we need to split the text which can be large into small chunks so that each chunk size is under the token limit. There are various text splitter classes in langchain that allows us to do this.


In [28]:
 # Say LLM token limit is 99, in that case we can do simple thing such as this
pageContent[0:99]

'Skip to main content\n\nSelect campus:UKDubai\n\n\n\nUniversity of Birmingham\n\nMenu\n\nA graphical represen'

In [34]:
# Well but we want complete words and want to do this for entire text, may be we can use Python's split funciton

words = pageContent.split(" ")

len(words)
# words 



136

In [35]:
chunks = []

s = ""
for word in words:
    s += word + " "
    if len(s)>99:
        chunks.append(s)
        s = ""
        
chunks.append(s)

In [32]:
chunks[0]

'Skip to main content\n\nSelect campus:UKDubai\n\n\n\nUniversity of Birmingham\n\nMenu\n\nA graphical representation '

Splitting data into chunks can be done in native python but it is a tidious process. Also if necessary, you may need to experiment with various delimiters in an iterative manner to ensure that each chunk does not exceed the token length limit of the respective LLM.

Langchain provides a better way through text splitter classes.

Using Text Splitter Classes from Langchain
CharacterTextSplitter
RecursiveCharacterTextSplitter

In [41]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator ="\n",
    chunk_size = 100,
    chunk_overlap =0
)

In [42]:
chunks = splitter.split_text(pageContent) #pageContent :  text
len(chunks)

Created a chunk of size 279, which is longer than the specified 100


14

In [43]:
for chunk in chunks:
    print(len(chunk))


72
73
46
279
84
93
78
76
72
94
88
89
91
94


As you can see, all though we gave 100 as a chunk size since the split was based on \n, it ended up creating chunks that are bigger than size 200.

Another class from Langchain can be used to recursively split the text based on a list of separators. This class is RecursiveTextSplitter. Let's see how it works

In [44]:
pageContent

'Skip to main content\n\nSelect campus:UKDubai\n\n\n\nUniversity of Birmingham\n\nMenu\n\nA graphical representation of a super computer\n\nSchool of Computer Science\n\nA graphical representation of a super computer\n\nBreaking new ground in the theory and practice of computational systems and their applications, the School of Computer Science is a progressive, inclusive department, providing specialist teaching and conducting world-leading research in fundamental and applied computer science.\n\nStudy with us\n\nAbout Computer Science\n\nLearn about our school\n\nPeople\n\nView our profiles\n\nUniversity of Birmingham\n\nEdgbaston\n\nBirmingham, B15 2TT\n\nUnited Kingdom\n\nTel:+44 (0)121 414 3344\n\nUniversity of Birmingham Instagram page\n\nUniversity of Birmingham LinkedIn page\n\nUniversity of Birmingham Twitter page\n\nUniversity of Birmingham Facebook page\n\nUniversity of Birmingham TikTok page\n\nUniversity of Birmingham Weibo page\n\nUniversity of Birmingham We Chat\n\nUnive

In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],  # List of separators based on requirement (defaults to ["\n\n", "\n", " "])
    chunk_size = 100,  # size of each chunk created
    chunk_overlap  = 0,  # size of  overlap between chunks in order to maintain the context
    length_function = len  # Function to calculate size, currently we are using "len" which denotes length of string however you can pass any token counter)
)


In [60]:
chunks= r_splitter.split_text(pageContent)
len(chunks)
chunks #printing chuncks of pageContent ... 


77

In [61]:
for chunk in chunks : 
            print(len(chunk))


77
74
46
99
96
82
88
97
79
77
73
96
94
92
94
97


how RecursiveCharacterTextSplitter works ...
Recursive text splitter uses a list of separators, i.e. separators = ["\n\n", "\n", "."]

first_split = text.split("\n\n")[0]
len(first_split)

So now it will first split using \n\n and then if the resulting chunk size is greater than the chunk_size parameter which is 100 in our case, then it will use the next separator which is \n


second_split = first_split.split("\n")[0]
len(second_split)

Third split exceeds chunk size 200. Now it will further try to split that using the third separator which is ' ' (space)

second_split[2]

When you split this using space (i.e. second_split[2].split(" ")), it will separate out each word and then it will merge those chunks such that their size is close to 100{“smart packing” not random merging}